In [1]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "many2019establishing")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "01_manyprimates_pilot_merged_data_v2.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
import pyreadstat

df = pd.read_csv(complete_path_1)

df['study_id']="many2019establishing"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)


In [3]:
df = df[df.site=='leipzig zoo']
df[['ape','site']] = df['subject_site'].str.split('_',expand=True)

In [4]:
df.rename(columns={"species": "species_original"}, inplace=True)
# df.columns

In [5]:
comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)
df['ape'] = df['ape'].str.rstrip()
for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)

comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
df= df.merge(apedf,left_on='ape', right_on='name', how='left')
# df.columns
df.rename(columns={"ape": "participant",
                   "age":"age_in_years"}, inplace=True)

In [6]:
many2019establishing_standardized=df[['study_id', 'participant','age_in_years',  'sex', 'species', 
        'session', 'trial', 'test_situation', 'task_experience', 'block',  'cup_distance', 'board_size',
        'delay', 'hiding_location', 'pick', 'correct', 
       'life_expectancy', 'norm_age', ]]
comp_out_path_stand = os.path.join(out_pathway, 'many2019establishing_standardized.csv')
many2019establishing_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)

names =many2019establishing_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
many2019establishing_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'many2019establishing_glossary.csv')
many2019establishing_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)

